In [233]:
# 08/2025
# Load distance matrices, generate isomap/Umap
# Join the above generated shape info with Database-specific infomation
#     add function_info to make complete_info
# Verifications
# Save the csv file if needed

In [234]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [235]:
#######################################  load distances  #####################################
curProject = 'amputee'
curRegion = 'CSSyl'  # CSSyl or CSpreCS
curRoot = 'C'  # 'C' or 'D'

##  reading from the basic analysis  ##
distance_path_min = rf'{curRoot}:\B_projWIP\proj_{curProject}\amputeeExtended_2024\{curRegion}\Isomap\minDist{curRegion}.txt'
distance_path_max = rf'{curRoot}:\B_projWIP\proj_{curProject}\amputeeExtended_2024\{curRegion}\Isomap\maxDist{curRegion}.txt'

try:
    minDist = pd.read_csv(distance_path_min, index_col=0, header=0)
    maxDist = pd.read_csv(distance_path_max, index_col=0, header=0)    
except FileNotFoundError as e:
    print(f"Error: {e}")

rows, cols = minDist.shape
print(f"Number of rows: {rows}")
print(f"Number of columns: {cols}")

Number of rows: 130
Number of columns: 130


In [236]:
################################    generation of Isomap    ##################################
# generation of isomaps using minDist and maxDist, all subjects, NO selection
# Define: outNameMin/Max, outFileNameMin/Max

from sklearn.manifold import Isomap
numDim = 3
numNeig = 10
genIsomap = False  ## !!!!!!!!!!!!!  default to False, True only after varification  !!!!!!!!!!!!!

outNameMin = 'isomapCmds'+curRegion+'k'+str(numNeig)+'d'+str(numDim)+'distmin'+'.txt'
outNameMax = 'isomapCmds'+curRegion+'k'+str(numNeig)+'d'+str(numDim)+'distmax'+'.txt'

outFileNameMin = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\form_measure\isomap\{outNameMin}'
outFileNameMax = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\form_measure\isomap\{outNameMax}'

subjNames = minDist.index
dimNames = np.arange(1,numDim+1)
iso_min = Isomap(n_neighbors=numNeig,n_components=numDim,metric='precomputed').fit_transform(minDist.values)
iso_min_DF = pd.DataFrame(iso_min,index=subjNames,columns=dimNames)
iso_max = Isomap(n_neighbors=numNeig,n_components=numDim,metric='precomputed').fit_transform(maxDist.values)
iso_max_DF = pd.DataFrame(iso_max,index=subjNames,columns=dimNames)

# SAVE Isomaps as csv
print(outFileNameMin)
print(outFileNameMax)
if genIsomap:
    iso_min_DF.to_csv(outFileNameMin,index_label='subjName')
    iso_max_DF.to_csv(outFileNameMax,index_label='subjName')


C:\B_projWIP\proj_amputee\Analysis_2025\form_measure\isomap\isomapCmdsCSSylk10d3distmin.txt
C:\B_projWIP\proj_amputee\Analysis_2025\form_measure\isomap\isomapCmdsCSSylk10d3distmax.txt


In [237]:
###############################   generation of UMAP   ################################
# UMAP for the original distance WITHOUT selection of subjects
import umap
import random

# to ensure that the results are always the same
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

writeUMAP = False     # default to False, True only after varification            ###   CHANGE   ###

typeDist = 'min'   # using minimum or maximum distance matrix                     ###   CHANGE   ###
n_comp = 1         # Change this to the desired number of dimensions: 1 or 2      ###   CHANGE   ###
n_neighbors = 5    # Change this to the desired number of neighbors: 5 or 30      ###   CHANGE   ###
min_dist = 0.2     # Change this to the desired minimum distance
# define df for UMAP generation
if (typeDist == 'min'):
    df = minDist    
if (typeDist == 'max'):
    df = maxDist        
    
###############################  define output file name  ############################
outName = 'dim'+str(n_comp)+'_'+typeDist+'_neig'+str(n_neighbors)+'_dist'+str(min_dist)+'.txt'
###############################  define output file path  ############################   

curTypeAnalysis = 'main_piece_analysis' # 'basic_analysis'  ##  !!!!!!!!   CHANGE  !!!!!!!!
outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\form_measure\umap_{curRegion}\{outName}'
##############################################################################################################################
# perform UMAP
reducer = umap.UMAP(metric='precomputed', n_components=n_comp,n_neighbors=n_neighbors,min_dist=min_dist,random_state=SEED)
embedding = reducer.fit_transform(df)

# Create a DataFrame for the embedding
if n_comp == 1:
    embedding_df = pd.DataFrame(embedding, columns=['UMAP1']) 
if n_comp == 2:
    embedding_df = pd.DataFrame(embedding, columns=['UMAP1', 'UMAP2'])
if n_comp == 3:
    embedding_df = pd.DataFrame(embedding, columns=['UMAP1', 'UMAP2', 'UMAP3'])
embedding_df.index = df.index

print(embedding_df)
print(outFileName)
# Save as csv
if writeUMAP:
    embedding_df.to_csv(outFileName,index_label='subjName')

                      UMAP1
subjName                   
LPC24_struct      -3.961019
LPC05_struct       1.284177
LPA30_struct       4.713331
LPC21_struct       1.567757
LPA32_struct      -2.894277
...                     ...
flip-RPC02_struct  0.830246
flip-RPC15_struct -3.853850
flip-RPA13_struct -5.255374
flip-RPC17_struct -0.699259
flip-RPC14_struct  5.100941

[130 rows x 1 columns]
C:\B_projWIP\proj_amputee\Analysis_2025\form_measure\umap_CSSyl\dim1_min_neig5_dist0.2.txt


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [238]:
##########################  Load shape measures, if generated and written above  #############################
curDistType = 'min'                                         ##############   CHANGE  ###############
curTypeAnalysis = 'form_measure'                            ##############   CHANGE  ###############
#curTypeAnalysis = 'main_piece_analysis' 

shape_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\isomap\isomapCmds{curRegion}k10d3dist{curDistType}.txt'
shapeU_1_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\umap_{curRegion}\dim1_{curDistType}_neig5_dist0.2.txt'
shapeU_2_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\umap_{curRegion}\dim1_{curDistType}_neig30_dist0.2.txt'
shapeU_3_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\umap_{curRegion}\dim2_{curDistType}_neig5_dist0.2.txt'
shapeU_4_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\umap_{curRegion}\dim2_{curDistType}_neig30_dist0.2.txt'


In [239]:
#######################  Load shape information  #######################
shape = shapeU_1 = shapeU_2 = shapeU_3 = shapeU_4 = None
try:
    shape = pd.read_csv(shape_path, index_col=0, header=0)
    #print(shape.head())
    shapeU_1 = pd.read_csv(shapeU_1_path, index_col=0, header=0)
    shapeU_2 = pd.read_csv(shapeU_2_path, index_col=0, header=0)
    shapeU_3 = pd.read_csv(shapeU_3_path, index_col=0, header=0)
    shapeU_4 = pd.read_csv(shapeU_4_path, index_col=0, header=0)

    shape.rename(columns={'1':'iso1'}, inplace=True)
    shape.rename(columns={'2':'iso2'}, inplace=True)
    shape.rename(columns={'3':'iso3'}, inplace=True)    
    shapeU_1.rename(columns={'UMAP1': 'UMAP1_U1'}, inplace=True)
    shapeU_2.rename(columns={'UMAP1': 'UMAP1_U2'}, inplace=True)
    shapeU_3.rename(columns={'UMAP1': 'UMAP1_U3', 'UMAP2': 'UMAP2_U3'}, inplace=True)
    shapeU_4.rename(columns={'UMAP1': 'UMAP1_U4', 'UMAP2': 'UMAP2_U4'}, inplace=True)
    #shapeU_4.rename(columns={'UMAP1': 'UMAP1_U4', 'UMAP2': 'UMAP2_U4','UMAP3': 'UMAP3_U4'}, inplace=True)
except FileNotFoundError as e:
    print(f"Error: {e}")

if all(df is not None for df in [shape, shapeU_1, shapeU_2, shapeU_3, shapeU_4]):
    shape_joined = shape.join([shapeU_1, shapeU_2, shapeU_3, shapeU_4])
    shape = shape_joined
    print(shape.head())
    print(shape.index)
else:
    print("One or more input files were missing — join was not performed.")
    

                  iso1      iso2      iso3  UMAP1_U1  UMAP1_U2  UMAP1_U3  \
subjName                                                                   
LPC24_struct -3.129764 -0.129110 -1.895557 -3.961019  3.855496  1.994226   
LPC05_struct -3.857513 -3.494893  1.386982  1.284177  3.298092  3.205916   
LPA30_struct  4.102761  1.443144 -0.599581  4.713331  7.585755  6.580620   
LPC21_struct -1.398569 -1.033002  0.397889  1.567757  3.023226  3.468465   
LPA32_struct -3.843092  2.432006  1.754709 -2.894277  1.895094  1.460530   

              UMAP2_U3   UMAP1_U4  UMAP2_U4  
subjName                                     
LPC24_struct  6.919744   8.511568  8.282360  
LPC05_struct  8.617145   8.910116  6.695747  
LPA30_struct  7.921068  11.009798  4.549885  
LPC21_struct  8.957874   9.184196  6.656776  
LPA32_struct  8.054934  10.108804  7.961687  
Index(['LPC24_struct', 'LPC05_struct', 'LPA30_struct', 'LPC21_struct',
       'LPA32_struct', 'LMA03_struct_nf', 'LPA12_struct', 'LPA16_struct',


In [240]:
###############################  Proecssing the shape file  ################################
# Add a 'SubjID' column, based on index, removing L, filp_R as prefix, remove _struct as postfix
# Add a 'Hemisphere' column (Left if index starts with L or Right if index starts with flip)

# Create SubjID from index, removing pre_fix and post_fix
shape['SubjID'] = (
    shape.index
    .to_series()  # convert index to a Series so we can use string operations
    .astype(str)   # convert index values to strings
    .str.replace(r'^(L|flip-R)', '', regex=True)   # remove 'L' or 'flip-R' at start
    .str.replace(r'_struct.*$', '', regex=True)    # remove '_struct' and anything after
)
shape['SubjID'] = shape['SubjID'].astype(str) # Make sure SubjID is string
shape['Study'] = 1 # Default Study = 1
shape.loc[shape['SubjID'].str.startswith('M'), 'Study'] = 2 # Assign 2 where SubjID starts with 'M'

# Create 'Hemisphere' based on index prefix
#shape['Hemisphere'] = shape.index.to_series().apply(
#shape['Hemisphere'] = shape[subjName].apply(
#    lambda x: 'Left' if x.startswith('L') else 'Right' if x.startswith('flip-R') else None
#)
shape['Hemisphere'] = shape.index.to_series().astype(str).apply(
    lambda x: 'Left' if x.startswith('L') else 'Right' if x.startswith('flip-R') else None
)



shape = shape.reset_index()

#print(shape.head())
print(shape.columns)
print(shape)

Index(['subjName', 'iso1', 'iso2', 'iso3', 'UMAP1_U1', 'UMAP1_U2', 'UMAP1_U3',
       'UMAP2_U3', 'UMAP1_U4', 'UMAP2_U4', 'SubjID', 'Study', 'Hemisphere'],
      dtype='object')
              subjName      iso1      iso2      iso3  UMAP1_U1  UMAP1_U2  \
0         LPC24_struct -3.129764 -0.129110 -1.895557 -3.961019  3.855496   
1         LPC05_struct -3.857513 -3.494893  1.386982  1.284177  3.298092   
2         LPA30_struct  4.102761  1.443144 -0.599581  4.713331  7.585755   
3         LPC21_struct -1.398569 -1.033002  0.397889  1.567757  3.023226   
4         LPA32_struct -3.843092  2.432006  1.754709 -2.894277  1.895094   
..                 ...       ...       ...       ...       ...       ...   
125  flip-RPC02_struct -2.283779  0.827991  2.133399  0.830246  2.612782   
126  flip-RPC15_struct -3.797792 -0.618864  0.383333 -3.853850  3.433703   
127  flip-RPA13_struct  0.995838  7.773048 -4.150236 -5.255374  0.143787   
128  flip-RPC17_struct -0.688494  5.321493 -1.216060 -0.699259

In [241]:
##################################### Prepare Database-specific infomation ####################################
# Load anatomical and functional info 
shape_info_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\{curProject}_INFO_result\allinfoAmputeeJoy.csv'
function_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\{curProject}_INFO_result\Functional_plasticity.csv'
shape_info = None
function_info = None

try:
    shape_info = pd.read_csv(shape_info_path, index_col=0, header=0,sep=',')
    function_info = pd.read_csv(function_path, index_col=0, header=0,sep=',')
except FileNotFoundError as e:
    print(f"Error: {e}")

if shape_info is not None:  # Proceed only if loading was successful
#    shape_info['SubjID'] = shape_info.index    # Make current index as a new column called SubjID
    shape_info['Group_num'] = shape_info['Group'].map({'CTR': 0, 'CONG': 1,'AMP':2})    # Create the new column 'Category_num'
if function_info is not None:  
    function_info['Study'] = function_info.index    # Make current index as a new column called study
    function_info.set_index('subjID', inplace=True)    # Make SubjID column as index, not keeping it as a separate column   
    function_info = function_info.assign(Group='CONG', Group_num=1) 
    function_info.rename(columns={'gender': 'Gender_function'}, inplace=True) # keep the gender info in function for verification only
    function_info.index.name = 'SubjID'  # By default subjID, here for merge rename it to agree between the two df's
    function_info = function_info.rename(columns={'amp.side': 'AmpSide_function'})
else:
    print("Info could not be loaded. Aborting further steps.")

print(shape_info.columns)
#print(shape_info.head())
#print("Original data types:\n", shape_info.dtypes)

print(function_info.columns)
#print(function_info)
#print("Original data types:\n", function_info.dtypes)

print(len(shape_info))
print(len(function_info))

print(shape_info.index.name)
print(function_info.index.name)

Index(['Gender', 'AgeScan', 'AgeLimbLoss', 'Group', 'AmpSide', 'DominantHand',
       'Group_num'],
      dtype='object')
Index(['Gender_function', 'AmpSide_function', 'birthyear', 'Prosthesisusage',
       'Stumpusage', 'amputatio level', 'Cos', 'Fun', 'Myo', 'missinghand',
       'intacthand', 'residualarm', 'intactarm', 'lips', 'feet', 'Study',
       'Group', 'Group_num'],
      dtype='object')
65
24
SubjID
SubjID


In [242]:
##################  Merging the ori shape_info with the function_info to make complete_info  ###################

#complete_info = shape_info.merge(function_info,on=['Group', 'Group_num'],left_index=True,right_index=True,how='left')
complete_info = shape_info.merge(function_info, on=['SubjID', 'Group','Group_num'], how='left')
complete_info = complete_info.reset_index()   # make 'SubjID' a column
complete_info['Study'] = complete_info['Study'].fillna(1)

"""
# Move the subjID (function_info) column next to SubjID (shape_info) for visual inspection
cols = list(complete_info.columns)               # current column order
cols.remove('subjID')                    # take out colB
insert_pos = cols.index('SubjID') + 1    # position right after colA
cols.insert(insert_pos, 'subjID')        # insert colB in the right place
complete_info = complete_info[cols]                          # reorder DataFrame

# Rename subjID (function_info) to SubjID_function, SubjID (amputee_info) to SubjID_amputeeS
complete_info.rename(columns={'subjID': 'SubjID_function'}, inplace=True)
complete_info.rename(columns={'SubjID': 'SubjID_amputee'}, inplace=True)
"""
print(complete_info.columns)
#print(complete_info.head())
#print("Original data types:\n", complete_info.dtypes)
print(len(complete_info))
#with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#    print(complete_info)
#    print(shape)

Index(['SubjID', 'Gender', 'AgeScan', 'AgeLimbLoss', 'Group', 'AmpSide',
       'DominantHand', 'Group_num', 'Gender_function', 'AmpSide_function',
       'birthyear', 'Prosthesisusage', 'Stumpusage', 'amputatio level', 'Cos',
       'Fun', 'Myo', 'missinghand', 'intacthand', 'residualarm', 'intactarm',
       'lips', 'feet', 'Study'],
      dtype='object')
65


In [243]:
##########################  Merging shape (Isomap/Umap) and complete_info  ###########################
merged_info = shape.merge(complete_info, on=['SubjID','Study'], how='left')  # left join, keep all the rows
merged_info.set_index('subjName', inplace=True)
#print(merged_info.head())
print(merged_info.columns)
print(len(merged_info))

Index(['iso1', 'iso2', 'iso3', 'UMAP1_U1', 'UMAP1_U2', 'UMAP1_U3', 'UMAP2_U3',
       'UMAP1_U4', 'UMAP2_U4', 'SubjID', 'Study', 'Hemisphere', 'Gender',
       'AgeScan', 'AgeLimbLoss', 'Group', 'AmpSide', 'DominantHand',
       'Group_num', 'Gender_function', 'AmpSide_function', 'birthyear',
       'Prosthesisusage', 'Stumpusage', 'amputatio level', 'Cos', 'Fun', 'Myo',
       'missinghand', 'intacthand', 'residualarm', 'intactarm', 'lips',
       'feet'],
      dtype='object')
130


In [244]:
#########################  Adding missing_hem column to the final complete_info  ##########################

# Default all to 'None'
merged_info['missing_hem'] = 'None'

# Set 'R' where AmpSide is 'L'
merged_info.loc[merged_info['AmpSide'] == 'L', 'missing_hem'] = 'R'

# Set 'L' where AmpSide is 'R'
merged_info.loc[merged_info['AmpSide'] == 'R', 'missing_hem'] = 'L'

In [245]:
###########################################  Verifications  ###########################################

In [246]:
print(merged_info.columns)

Index(['iso1', 'iso2', 'iso3', 'UMAP1_U1', 'UMAP1_U2', 'UMAP1_U3', 'UMAP2_U3',
       'UMAP1_U4', 'UMAP2_U4', 'SubjID', 'Study', 'Hemisphere', 'Gender',
       'AgeScan', 'AgeLimbLoss', 'Group', 'AmpSide', 'DominantHand',
       'Group_num', 'Gender_function', 'AmpSide_function', 'birthyear',
       'Prosthesisusage', 'Stumpusage', 'amputatio level', 'Cos', 'Fun', 'Myo',
       'missinghand', 'intacthand', 'residualarm', 'intactarm', 'lips', 'feet',
       'missing_hem'],
      dtype='object')


In [247]:
ctl_counts = shape_info['Group_num'].value_counts()
print(ctl_counts)
ctl_counts = function_info['Group_num'].value_counts()
print(ctl_counts)
ctl_counts = merged_info['Group_num'].value_counts()
print(ctl_counts)

Group_num
1    25
0    24
2    16
Name: count, dtype: int64
Group_num
1    24
Name: count, dtype: int64
Group_num
1    50
0    48
2    32
Name: count, dtype: int64


In [248]:
####################  Display the values of a specific column  ####################
print("Values in the 'iso1' column:")
print(merged_info['iso1'])

####################  Display the rows with a specific column value  ####################
cer = merged_info[merged_info['Group'] == 1]
#print(cer['Category'])

####################  Get a summary of statistics  ####################
summary_stats = merged_info['iso1'].describe()
print("\nSummary statistics for the 'isomap1' column:")
print(summary_stats)

####################  Detect null values in a specific column  ###################
null_values = merged_info['Group'].isnull()
#print(null_values)

####################  Filter rows where the specified column has null values  #####################
null_rows = merged_info[merged_info['iso1'].isnull()]
#print(null_rows['1'])

####################  Count the number of null values in a specific column  ######################
null_count = merged_info['iso1'].isnull().sum()
print(f"Number of null values in selected column: {null_count}")


Values in the 'iso1' column:
subjName
LPC24_struct        -3.129764
LPC05_struct        -3.857513
LPA30_struct         4.102761
LPC21_struct        -1.398569
LPA32_struct        -3.843092
                       ...   
flip-RPC02_struct   -2.283779
flip-RPC15_struct   -3.797792
flip-RPA13_struct    0.995838
flip-RPC17_struct   -0.688494
flip-RPC14_struct    2.262438
Name: iso1, Length: 130, dtype: float64

Summary statistics for the 'isomap1' column:
count    1.300000e+02
mean    -1.366428e-16
std      3.105392e+00
min     -5.370157e+00
25%     -2.534199e+00
50%     -3.600649e-01
75%      2.275539e+00
max      7.290825e+00
Name: iso1, dtype: float64
Number of null values in selected column: 0


In [282]:
################################  Saving csv files if needed  #################################
file_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curRegion}_combined_{curDistType}.csv'
print(file_path)

# Write the DataFrame to a CSV file
#merged_info.to_csv(file_path, index=True)

################################  Test read the CSV file back into a DataFrame  ################################
#df_loaded = pd.read_csv(file_path)
#print(len(df_loaded))
#print("Data types:\n", df_loaded.dtypes)

C:\B_projWIP\proj_amputee\Analysis_2025\CSSyl_combined_min.csv
